# CheXpert — 95/5 patient-grouped split (`02_train.csv` / `02_val.csv`)

Same logic as `02_splits_dataset.ipynb` (frontal-only, patient-grouped, `SEED=42`,
reproducible), but holds out **5%** of patients for in-loop validation instead of
10% — 10% was more validation than needed, so this frees ~5% more rows for training.

This notebook ONLY splits the dataset and prints the details before/after. No EDA,
no Dataset/preprocessing (those live in `02_splits_dataset.ipynb`).

In [1]:
# Load train.csv, keep frontal only, parse patient id from the path (same as 02_splits_dataset.ipynb)
import numpy as np
import pandas as pd
from pathlib import Path

DATA_ROOT = Path(r"D:\CheXpert-v1.0-small")                              # source CSVs
OUT_DIR   = Path(r"d:\My Projects\chest-xray-bench\data\chexpert")       # where split CSVs go
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_TASKS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

train_df = pd.read_csv(DATA_ROOT / "train.csv")
train_df = train_df[train_df["Frontal/Lateral"] == "Frontal"].copy()
train_df["patient_id"] = train_df["Path"].str.extract(r"(patient\d+)", expand=False)

print("frontal train rows:", len(train_df))
print("unique patients   :", train_df["patient_id"].nunique())
print("OUT_DIR           :", OUT_DIR)

frontal train rows: 191027
unique patients   : 64534
OUT_DIR           : d:\My Projects\chest-xray-bench\data\chexpert


In [2]:
# Patient-grouped split: carve a 5% in-loop validation set out of train (frozen, reproducible)
from sklearn.model_selection import GroupShuffleSplit

VAL_FRAC = 0.05     # fraction of PATIENTS held out for in-loop validation (was 0.10 in the 01_ split)
SEED     = 42       # fixed -> split is reproducible/frozen across all experiments

def uones_rate(df):  # U-Ones positive rate per target task (pos + uncertain), % of rows
    return {t: round(100 * df[t].isin([1, -1]).mean(), 2) for t in TARGET_TASKS}

print("=" * 60)
print("BEFORE split")
print(f"  total frontal rows : {len(train_df)}")
print(f"  unique patients    : {train_df['patient_id'].nunique()}")
print(f"  U-Ones %pos        : {uones_rate(train_df)}")
print("=" * 60)

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
tr_idx, va_idx = next(gss.split(train_df, groups=train_df["patient_id"]))
tr = train_df.iloc[tr_idx].copy()
va = train_df.iloc[va_idx].copy()

p_tr, p_va = set(tr["patient_id"]), set(va["patient_id"])
print("AFTER split")
print(f"  VAL_FRAC={VAL_FRAC}  SEED={SEED}")
print(f"  train rows : {len(tr):>7d}  ({100*len(tr)/len(train_df):.1f}%)   patients: {len(p_tr)}")
print(f"  val   rows : {len(va):>7d}  ({100*len(va)/len(train_df):.1f}%)   patients: {len(p_va)}")
print(f"  shared patients (must be 0): {len(p_tr & p_va)}")
print(f"  train U-Ones %pos : {uones_rate(tr)}")
print(f"  val   U-Ones %pos : {uones_rate(va)}")
print("=" * 60)

BEFORE split
  total frontal rows : 191027
  unique patients    : 64534
  U-Ones %pos        : {'Atelectasis': np.float64(31.19), 'Cardiomegaly': np.float64(15.75), 'Consolidation': np.float64(19.56), 'Edema': np.float64(32.19), 'Pleural Effusion': np.float64(45.27)}
AFTER split
  VAL_FRAC=0.05  SEED=42
  train rows :  181570  (95.0%)   patients: 61307
  val   rows :    9457  (5.0%)   patients: 3227
  shared patients (must be 0): 0
  train U-Ones %pos : {'Atelectasis': np.float64(31.19), 'Cardiomegaly': np.float64(15.78), 'Consolidation': np.float64(19.56), 'Edema': np.float64(32.25), 'Pleural Effusion': np.float64(45.29)}
  val   U-Ones %pos : {'Atelectasis': np.float64(31.23), 'Cardiomegaly': np.float64(15.27), 'Consolidation': np.float64(19.48), 'Edema': np.float64(31.12), 'Pleural Effusion': np.float64(44.97)}


In [3]:
# Write the frozen split CSVs to data/chexpert/  (02_train.csv, 02_val.csv)
train_out = OUT_DIR / "02_train.csv"
val_out   = OUT_DIR / "02_val.csv"

print("WRITING")
print(f"  {train_out}  <-  {len(tr)} rows, {len(tr.columns)} cols")
print(f"  {val_out}  <-  {len(va)} rows, {len(va.columns)} cols")
print(f"  columns: {list(tr.columns)}")

tr.to_csv(train_out, index=False)
va.to_csv(val_out, index=False)

# Read back and verify integrity
tr_chk = pd.read_csv(train_out)
va_chk = pd.read_csv(val_out)
p_tr_chk = set(tr_chk["patient_id"]); p_va_chk = set(va_chk["patient_id"])

print("\nVERIFY (read back from disk)")
print(f"  02_train.csv : {len(tr_chk)} rows  (matches: {len(tr_chk)==len(tr)})   patients: {len(p_tr_chk)}")
print(f"  02_val.csv   : {len(va_chk)} rows  (matches: {len(va_chk)==len(va)})   patients: {len(p_va_chk)}")
print(f"  shared patients across saved files (must be 0): {len(p_tr_chk & p_va_chk)}")
print(f"  total saved rows: {len(tr_chk)+len(va_chk)}  (== frontal train {len(train_df)}: {len(tr_chk)+len(va_chk)==len(train_df)})")

WRITING
  d:\My Projects\chest-xray-bench\data\chexpert\02_train.csv  <-  181570 rows, 20 cols
  d:\My Projects\chest-xray-bench\data\chexpert\02_val.csv  <-  9457 rows, 20 cols
  columns: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'patient_id']

VERIFY (read back from disk)
  02_train.csv : 181570 rows  (matches: True)   patients: 61307
  02_val.csv   : 9457 rows  (matches: True)   patients: 3227
  shared patients across saved files (must be 0): 0
  total saved rows: 191027  (== frontal train 191027: True)
